# Vision Mamba

<div>
    <img src="../images/VISIONMAMBA.png" width="900">
</div>

## Vim Block Flow

The **Vision Mamba (Vim)** block extends the original Mamba by introducing **bidirectional sequence modeling**, allowing image patches to gather context from both directions.

The processing pipeline is:

1. **LayerNorm:** Normalize the input token sequence.
2. **Linear Projection:** Split the features into **x** (SSM branch) and **z** (gating branch), expanding the dimension from **D → E**.
3. **Forward & Backward Processing:** Process the **x** branch independently in the forward and backward directions.
4. **1-D Convolution:** Extract local spatial features before the SSM.
5. **Parameter Projection:** Project the convolution output to obtain the input-dependent parameters **B(x)**, **C(x)**, and **Δ(x)** (denoted as **Bₒ**, **Cₒ**, and **Δₒ** in the paper, where *o* indicates the processing direction).
6. **Discretization:** Use **Δₒ** to discretize the continuous SSM parameters, producing the discrete matrices **Āₒ** and **B̄ₒ** used by the Selective SSM.
7. **Selective SSM:** Compute the forward and backward hidden states using **Āₒ**, **B̄ₒ**, and **Cₒ**.
8. **Gating:** Modulate both outputs using the **z** branch.
9. **Fusion:** Sum the forward and backward outputs.
10. **Output Projection:** Project **E → D** and apply the residual connection.

```text
Input
  │
LayerNorm
  │
Linear Projection
 ┌───────┐
 │       │
 x       z
 │       │
 ├──► Forward Conv1d ─► Forward SSM ─┐
 │                                   × z
 └──► Backward Conv1d ─► Backward SSM┘
                │
           Sum Outputs
                │
      Linear Projection (E → D)
                │
        Residual Connection
                │
             Output
```

> **Key idea:** Unlike the original Mamba, which performs **unidirectional** sequence modeling, Vision Mamba processes image patches in **both forward and backward directions**, while preserving Mamba's selective mechanism (**A** fixed, **B**, **C**, and **Δ** input-dependent).

## Results

<div>
    <img src="../images/VMAMBA_T1.png" width="400">
    <img src="../images/VMAMBA_FIG1.png" width="900">
</div>

### ImageNet-1K Results

- **Outperforms ConvNets:** Vim significantly surpasses ResNet with a similar number of parameters (e.g., **+4.1% Top-1** over ResNet-50).

- **Competitive with Transformers:** Vim achieves equal or better accuracy than ViT and DeiT while using comparable or fewer parameters.

- **More parameter-efficient:** Compared with S4ND-ViT, Vim reaches similar accuracy using **about 3× fewer parameters**.

- **Scales to longer sequences:** After fine-tuning on longer image sequences, all Vim variants improve further, demonstrating good adaptability to higher resolutions.

- **Higher computational efficiency:** Thanks to its **linear sequence complexity**, Vim becomes increasingly faster and more memory-efficient as image resolution increases.

- **High-resolution advantage:** At **1248×1248** resolution, Vim is **2.8× faster** than DeiT while reducing **GPU memory usage by 86.8%**.

> **Key takeaway:** Vim not only improves classification accuracy over CNN-, Transformer-, and previous SSM-based backbones, but also offers substantially better speed and memory efficiency for high-resolution images.